Business Questions

Q1: Delivery Performance

What is the late delivery rate, and which shipping modes or regions have the highest delays?

Q2: Profitability Analysis

Which markets, regions, and product categories generate the most profit and sales?

Q3: Shipping Efficiency

Which shipping mode is the most efficient in terms of cost, time, and delivery success?

##Import required liberaries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding= 'latin1')
df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


##Understanding Data

In [3]:
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
       'Product De

In [4]:
df.shape

(180519, 53)

In [5]:
df.isnull().sum()[df.isnull().sum()>0]

,0
Customer Lname,8
Customer Zipcode,3
Order Zipcode,155679
Product Description,180519


In [6]:
df['Delivery Status'].value_counts()

,count
Delivery Status,
Late delivery,98977
Advance shipping,41592
Shipping on time,32196
Shipping canceled,7754


In [7]:
df['Shipping Mode'].value_counts()

,count
Shipping Mode,
Standard Class,107752
Second Class,35216
First Class,27814
Same Day,9737


##Cleaning

In [8]:
# Drop unnecessary columns
df = df.drop(columns= ['Order Zipcode', 'Product Description', 'Product Image',
                       'Customer Password', 'Customer Email'])

In [9]:
# Drop rows with missing values
df = df.dropna(subset=['Customer Lname', 'Customer Zipcode'])

In [11]:
# convert dates to datetime
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])

In [13]:
df.shape

(180508, 48)

In [14]:
df.isnull().sum()[df.isnull().sum()>0]

,0


#Delivery Performance

In [17]:
# delievery status percentage
delivery_pct = df['Delivery Status'].value_counts(normalize=True) * 100
delivery_pct.round(2)

,proportion
Delivery Status,
Late delivery,54.83
Advance shipping,23.04
Shipping on time,17.83
Shipping canceled,4.30


In [18]:
# add delay column
df['shipping delay'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['shipping delay'].describe()

,shipping delay
count,180508.000000
mean,0.565836
std,1.490976
min,-2.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,4.000000


In [21]:
# shipping mode / delay
df.groupby('Shipping Mode')['shipping delay'].mean().round(2).sort_values(ascending=False)

,shipping delay
Shipping Mode,
Second Class,1.99
First Class,1.00
Same Day,0.48
Standard Class,-0.00


In [22]:
# delay based on market
df.groupby('Market')['shipping delay'].mean().round(2).sort_values(ascending=False)

,shipping delay
Market,
Europe,0.57
USCA,0.57
Pacific Asia,0.57
Africa,0.56
LATAM,0.56


In [23]:
# delay percentage
df.groupby('Market')['Late_delivery_risk'].mean().mul(100).round(2).sort_values(ascending=False)

,Late_delivery_risk
Market,
Europe,55.21
Pacific Asia,55.05
USCA,54.80
Africa,54.59
LATAM,54.36


#Profitability Analysis

In [24]:
# most profitable market
df.groupby('Market')['Order Profit Per Order'].mean().round(2).sort_values(ascending=False)

,Order Profit Per Order
Market,
Europe,23.26
USCA,21.87
LATAM,21.77
Africa,21.70
Pacific Asia,20.79


In [25]:
# most profitable categories
df.groupby('Category Name')['Sales'].sum().round(2).sort_values(ascending=False).head(10)

,Sales
Category Name,
Fishing,6929653.69
Cleats,4431942.78
Camping & Hiking,4118425.57
Cardio Equipment,3694843.20
Women's Apparel,3147800.00
Water Sports,3113844.68
Men's Footwear,2891757.66
Indoor/Outdoor Games,2888993.91
Shop By Sport,1309522.04


In [28]:
# categorz performance: sales vs profit efficiency
category_analysis = df.groupby('Category Name').agg(total_sales=('Sales', 'sum'),
                                                    avg_profit=('Order Profit Per Order','mean')
                                                    ).round(2).sort_values('total_sales', ascending=False).head(10)

In [29]:
category_analysis

,total_sales,avg_profit
Category Name,,
Fishing,6929653.69,43.65
Cleats,4431942.78,20.15
Camping & Hiking,4118425.57,31.14
Cardio Equipment,3694843.20,30.67
Women's Apparel,3147800.00,16.66
Water Sports,3113844.68,20.92
Men's Footwear,2891757.66,14.02
Indoor/Outdoor Games,2888993.91,16.50
Shop By Sport,1309522.04,11.82


# Shipping Efficiency

In [33]:
# order vlolume & cancellation rate by shipping mode
df.groupby('Shipping Mode').agg(
    total_orders=('Order Id','count'),
    avg_delay=('shipping delay', 'mean'),
    late_delivery_rate= ('Late_delivery_risk', 'mean')
).round(2).sort_values('late_delivery_rate', ascending=False)

,total_orders,avg_delay,late_delivery_rate
Shipping Mode,,,
First Class,27812,1.00,0.95
Second Class,35214,1.99,0.77
Same Day,9737,0.48,0.46
Standard Class,107745,-0.00,0.38


# Export Data

In [34]:
df.to_csv('supply_chain.cleaned.csv', index=False)
print('file exported successfully')

file exported successfully


# Supply Chain Performance Analysis
## DataCo Global Supply Chain Dataset
### Tools: Python (Pandas) | Power BI

## Project Objective
Analyze supply chain performance across delivery, profitability,
and shipping efficiency to identify operational bottlenecks
and business opportunities.

**Business Questions:**
- Q1: What is the late delivery rate, and which shipping modes have the highest delays?
- Q2: Which markets and product categories generate the most profit?
- Q3: Which shipping mode is the most efficient?

## Key Findings

### Q1: Delivery Performance
- 54.8% of all orders are delivered late — a systemic issue across all markets
- Late delivery risk is consistent globally (~55%) with no single region as outlier

### Q2: Profitability
- Europe is the most profitable market ( 23.26 \$ avg profit per order)
- Fishing leads in total sales (USD 6.9M \$) but Computers is the most efficient (157 \$ profit/order)
- ABC Analysis reveals high-volume vs high-margin category trade-offs

### Q3: Shipping Efficiency
- Standard Class has the lowest late delivery rate (38%) despite being the slowest
- First Class has the highest late delivery rate (95%) — a critical operational mismatch
- Same Day is the most balanced option (46% late, 0.48 days delay)

## Recommendations
1. Investigate First Class fulfillment process — 95% late rate is unacceptable for a premium service
2. Promote Standard Class for cost-sensitive customers — most reliable option
3. Focus marketing investment on Computers category — highest profit per order (157 \$)
4. Late delivery is a global operational issue — requires process-level intervention, not regional fixes